In this part of the project, we clean the data that was generated in the ingestion step. When you run this cell, the train and test files are going to be loaded, and they will be cleaned according to the instructions that are provided in the paper. At first, all rows with Null values will be removed (6.7 percent of the data). Then, the length of every sequence is checked, and the ones that are shorter than 35 points will be removed. Finally, the sensor numbers will get normalized, so they have a zero mean and a standard deviation of 1. The cleaned data will be saved as new files (with the word “cleaned” at the start of their names) so they can be used for the model implementation step.

In [ ]:
import pandas as pd
import pickle
import os
import numpy as np

# path for where our pkl files are stored
# going up from notebooks folder into data/processed
data_dir = "../data/processed/"
# we only need to clean these two sensor data files
files = ["train.pkl", "test.pkl"]

def my_preprocessing_function(df):
    # first we drop null rows
    # the reference paper says about 6.7% of data is removed this way
    df = df.dropna()
    
    # now we remove sequences that are too short
    # the paper says to remove sequences shorter than 35
    if 'sequence_id' in df.columns:
        counts = df['sequence_id'].value_counts()
        # find the ids that have at least 35 rows
        keep_ids = counts[counts >= 35].index
        df = df[df['sequence_id'].isin(keep_ids)]
    
    # finally we normalize the sensor columns 
    # we need zero mean and unit variance for the MLP model
    for col in df.columns:
        # we skip the id and label columns
        if col not in ['sequence_id', 'step', 'behavior']:
            if df[col].dtype == np.float64 or df[col].dtype == np.int64:
                m = df[col].mean()
                s = df[col].std()
                # standard formula: (x - mean) / std
                df[col] = (df[col] - m) / s
                
    return df

# loop to process both train and test files
for f_name in files:
    full_path = data_dir + f_name
    
    if os.path.exists(full_path):
        # open the file from ingestion
        with open(full_path, 'rb') as f:
            raw_data = pickle.load(f)
            
        # run the cleaning
        processed_data = my_preprocessing_function(raw_data)
        
        # save it for the models notebook
        output_name = data_dir + "cleaned_" + f_name
        with open(output_name, 'wb') as f:
            pickle.dump(processed_data, f)
            print("Successfully cleaned and saved:", output_name)
    else:
        print("Could not find the file:", f_name)

Successfully cleaned and saved: ../data/processed/cleaned_train.pkl
Successfully cleaned and saved: ../data/processed/cleaned_test.pkl
